### Rename every html attachment to f'{citekey}.html', where citekey is the betterbibtex citation key.

Started from [here](https://www.perplexity.ai/search/write-the-python-to-go-through-Hc0RTYl_RoG849gmN99MnA#3).

In [1]:
from icecream import ic
import pathlib as pl
#from collections.abc import Iterable
from pyzotero import zotero
#from collections import defaultdict
import pandas as pd
import refwrangle as rfw
#import matplotlib.pyplot as plt
import re

ModuleNotFoundError: No module named 'refwrangle'

In [ ]:
ic(rfw.library_id, rfw.library_type, rfw.api_key)

In [ ]:
#!/usr/bin/env python3

from pyzotero import zotero

# Point to your local Zotero 7 API. 
# By default, Zotero 7 may listen at "http://127.0.0.1:23119/zotero" 
# with a library_id of "0" for your personal library.
ZOTERO_LOCAL_URL = "http://127.0.0.1:23119/zotero"
LIBRARY_ID = "0"             # Zotero 'user' library ID, often "0" locally
API_KEY = ""                 # If needed, supply a local API key (may be blank for local read-only)
zot = zotero.Zotero(LIBRARY_ID, "user", API_KEY, url=ZOTERO_LOCAL_URL)

zot = zotero.Zotero(rfw.library_id, rfw.library_type, rfw.api_key)


In [ ]:

# Regex to capture pinned citekeys. Adjust if your pinned format differs.
citekey_pattern = re.compile(r"(?:bibtex|biblatex|citekey):\s*(\S+)")

# Retrieve all attachments in the library. 
# If your library is large, consider using zot.everything() or a paged approach.
attachments = zot.items(itemType="attachment", limit=100)

for attachment in attachments:
    data = attachment["data"]
    # Check if it's an HTML file by contentType or filename.
    ct = data.get("contentType", "")
    fn = data.get("filename", "")
    if ct == "text/html" or fn.lower().endswith(".html"):
        parent_key = data.get("parentItem")
        if not parent_key:
            continue

        # Get the parent item and parse its Extra field for the pinned citekey
        parent_item = zot.item(parent_key)
        extra_field = parent_item["data"].get("extra", "")
        match = citekey_pattern.search(extra_field)
        if match:
            citekey = match.group(1)
            # Rename the HTML attachment to f'{citekey}.html'
            data["filename"] = f"{citekey}.html"

            # Send the update to Zotero
            zot.update_item(attachment)
            print(f"Renamed attachment {attachment['key']} to {citekey}.html")
